# ML-06 — Signal Audit: Do the Flags Hold?

This notebook audits three signals that a content-decline baseline might lean on,
gives each a one-word verdict, and tests one signal behind a real FlyRank flag.

> **Skills loaded:** `auditing-signals` + `flyrank/flyrank-data`
> 
> **Lane:** Binary classification — predicting `is_declining` (impressions dropped ≥ 20%)

## 1. Distributions

*Look before deciding: distributions of key fields. Note the heavy tails.*

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

print(f"Dataset: {len(df):,} rows × {len(df.columns)} columns")
print(f"Base rate: {df['is_declining'].mean()*100:.1f}% declining")
print()

key_fields = ["impressions_90d", "days_since_last_update", "avg_position", "content_age_days"]

print("Distributions of key fields:")
print("=" * 70)
print(df[key_fields].describe().round(1).to_string())
print()
print("Observations:")
print("  • impressions_90d: heavily right-skewed (mean 5,200 vs median 731).")
print("    A few pages dominate; most have modest visibility.")
print("  • days_since_last_update: median 20 days, 75th pct at 104.")
print("    Most pages are relatively fresh; a tail extends to 373 days.")
print("  • avg_position: median 10.8, right-skewed. 1,205 rows have 0 (no data).")
print("  • content_age_days: ranges 90–564, fairly spread. All ≥ 90 in this slice.")

Dataset: 30,000 rows × 45 columns
Base rate: 54.2% declining

Distributions of key fields:
       impressions_90d  days_since_last_update  avg_position  content_age_days
count          30000.0                 30000.0       30000.0           30000.0
mean            5200.4                    46.1          16.3             256.2
std            16838.0                    42.1          15.2             132.7
min                1.0                     1.0           0.0              90.0
25%               81.0                    20.0           6.2             132.0
50%              731.0                    20.0          10.8             236.0
75%             3615.2                   104.0          22.3             333.0
max           517715.0                   373.0         245.0             564.0

Observations:
  • impressions_90d: heavily right-skewed (mean 5,200 vs median 731).
    A few pages dominate; most have modest visibility.
  • days_since_last_update: median 20 days, 75th pct at 10

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal test #1: Staleness → Decline

**Claim:** Pages that haven't been updated recently decline more often.
This is linked to FlyRank's refresh flags, which use staleness to trigger review.

In [2]:
# Signal test 1: Staleness (freshness_tier)
# Claim: staler pages (longer since last update) have higher decline rates.

stale_table = df.groupby("freshness_tier").agg(
    n=("is_declining", "count"),
    n_declining=("is_declining", "sum"),
    decline_rate=("is_declining", "mean"),
).copy()
stale_table["decline_rate_pct"] = (stale_table["decline_rate"] * 100).round(1)

tier_order = ["0-30", "31-90", "91-180", "181+"]
stale_table = stale_table.reindex([t for t in tier_order if t in stale_table.index])

print("Staleness → Decline rate (by freshness_tier):")
print("=" * 55)
print(stale_table[["n", "n_declining", "decline_rate_pct"]].to_string())
print(f"\nBase rate: {df['is_declining'].mean()*100:.1f}%")
print()
print("Verdict: MIXED")
print("  Decline rate rises from 0-30 days (51.1%) through 91-180 days (61.1%),")
print("  supporting the staleness hypothesis in the main range. However, it drops")
print("  to 47.1% at 181+ days (n=174), suggesting that extremely stale pages may")
print("  have already stabilized at a low traffic level and can't decline further.")
print("  The signal is directionally useful for the 0–180 day range but non-monotonic.")

Staleness → Decline rate (by freshness_tier):
                    n  n_declining  decline_rate_pct
freshness_tier                                      
0-30            20480        10473              51.1
31-90             175          103              58.9
91-180           9171         5604              61.1
181+              174           82              47.1

Base rate: 54.2%

Verdict: MIXED
  Decline rate rises from 0-30 days (51.1%) through 91-180 days (61.1%),
  supporting the staleness hypothesis in the main range. However, it drops
  to 47.1% at 181+ days (n=174), suggesting that extremely stale pages may
  have already stabilized at a low traffic level and can't decline further.
  The signal is directionally useful for the 0–180 day range but non-monotonic.


### Signal test #2: Position → Decline

**Claim:** Pages with worse search positions decline more often.
This is linked to FlyRank's CTR-fix logic, which flags pages with poor position-relative CTR.

In [3]:
# Signal test 2: Position (position_tier)
# Claim: pages with worse avg_position have higher decline rates.
# Exclude avg_position == 0 ("no data", not position zero) per data dictionary.

pos_df = df[df["avg_position"] > 0].copy()
pos_table = pos_df.groupby("position_tier").agg(
    n=("is_declining", "count"),
    n_declining=("is_declining", "sum"),
    decline_rate=("is_declining", "mean"),
).copy()
pos_table["decline_rate_pct"] = (pos_table["decline_rate"] * 100).round(1)

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
pos_table = pos_table.reindex([t for t in tier_order if t in pos_table.index])

print("Position → Decline rate (by position_tier, excl. no_data):")
print("=" * 55)
print(pos_table[["n", "n_declining", "decline_rate_pct"]].to_string())
print(f"\nn (excl. no_data): {len(pos_df):,}")
print(f"Base rate: {df['is_declining'].mean()*100:.1f}%")
print()
print("Verdict: MIXED")
print("  Decline rate rises from top_3 (49.4%) through striking distance (61.0%),")
print("  suggesting that pages slipping off page 1 face the highest decline risk.")
print("  However, deep pages (>50) have the lowest rate (34.4%, n=1,319).")
print("  Interpretation: deep pages have already lost most traffic and can't decline")
print("  further in percentage terms. The signal is strongest in the page_1–striking")
print("  range, which is exactly where FlyRank's CTR-fix logic operates.")

Position → Decline rate (by position_tier, excl. no_data):
                   n  n_declining  decline_rate_pct
position_tier                                      
top_3           1116          551              49.4
page_1         11814         6730              57.0
striking        7304         4452              61.0
page_3_5        7242         4067              56.2
deep            1319          454              34.4

n (excl. no_data): 28,795
Base rate: 54.2%

Verdict: MIXED
  Decline rate rises from top_3 (49.4%) through striking distance (61.0%),
  suggesting that pages slipping off page 1 face the highest decline risk.
  However, deep pages (>50) have the lowest rate (34.4%, n=1,319).
  Interpretation: deep pages have already lost most traffic and can't decline
  further in percentage terms. The signal is strongest in the page_1–striking
  range, which is exactly where FlyRank's CTR-fix logic operates.


### Signal test #3: Visibility → Decline

**Claim:** Pages with more impressions are more likely to decline.
This is linked to the volume signal behind FlyRank's quick-win flag.

In [4]:
# Signal test 3: Visibility (impression_tier)
# Claim: more visible pages (higher impressions) have higher decline rates.

vis_table = df.groupby("impression_tier").agg(
    n=("is_declining", "count"),
    n_declining=("is_declining", "sum"),
    decline_rate=("is_declining", "mean"),
).copy()
vis_table["decline_rate_pct"] = (vis_table["decline_rate"] * 100).round(1)

tier_order = ["none", "low", "moderate", "good", "excellent"]
vis_table = vis_table.reindex([t for t in tier_order if t in vis_table.index])

print("Visibility → Decline rate (by impression_tier):")
print("=" * 55)
print(vis_table[["n", "n_declining", "decline_rate_pct"]].to_string())
print(f"\nBase rate: {df['is_declining'].mean()*100:.1f}%")
print()
print("Verdict: MIXED")
print("  Decline rate is highest for moderate-visibility pages (61.5%, n=10,469)")
print("  and good-visibility pages (58.6%, n=7,205), both above base rate.")
print("  But low-visibility (45.4%) and excellent (46.2%) are below base rate.")
print("  The pattern is an inverted U: mid-range visibility pages decline most,")
print("  while the very small and very large pages are more stable. This makes")
print("  sense — tiny pages have little to lose, and giants have strong demand.")

Visibility → Decline rate (by impression_tier):
                     n  n_declining  decline_rate_pct
impression_tier                                      
low              11248         5106              45.4
moderate         10469         6435              61.5
good              7205         4223              58.6
excellent         1078          498              46.2

Base rate: 54.2%

Verdict: MIXED
  Decline rate is highest for moderate-visibility pages (61.5%, n=10,469)
  and good-visibility pages (58.6%, n=7,205), both above base rate.
  But low-visibility (45.4%) and excellent (46.2%) are below base rate.
  The pattern is an inverted U: mid-range visibility pages decline most,
  while the very small and very large pages are more stable. This makes
  sense — tiny pages have little to lose, and giants have strong demand.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**FlyRank flag tested:** The refresh flag uses `days_since_last_update >= 180` combined
with `impressions_90d >= 500` to identify "stale visible pages" that need review
(see `scripts/02_baseline_score.py`, line 25–26).

**Assumption being tested:** Among pages with meaningful visibility (≥ 500 impressions),
staler pages decline more often.

In [5]:
# Flag-linked test: staleness among visible pages
# FlyRank's refresh flag: days_since_last_update >= 180 AND impressions_90d >= 500
#
# Since the 180-day threshold gives only n=17 visible+stale pages (below the
# ~50 sample-size floor), I test the broader pattern: does staleness predict
# decline among visible pages at the 91-day boundary (freshness_tier 91-180),
# which has solid sample sizes?

visible_df = df[df["impressions_90d"] >= 500].copy()
print(f"Visible pages (impressions ≥ 500): n = {len(visible_df):,}")
print(f"Base decline rate (visible): {visible_df['is_declining'].mean()*100:.1f}%")
print()

# Test at multiple thresholds
print("Decline rate by staleness tier among VISIBLE pages (impr ≥ 500):")
print("=" * 60)
vis_stale = visible_df.groupby("freshness_tier").agg(
    n=("is_declining", "count"),
    n_declining=("is_declining", "sum"),
    decline_rate=("is_declining", "mean"),
).copy()
vis_stale["decline_rate_pct"] = (vis_stale["decline_rate"] * 100).round(1)
tier_order = ["0-30", "31-90", "91-180", "181+"]
vis_stale = vis_stale.reindex([t for t in tier_order if t in vis_stale.index])
print(vis_stale[["n", "n_declining", "decline_rate_pct"]].to_string())
print()

# The 180-day threshold specific test
visible_df["stale_180"] = (visible_df["days_since_last_update"] >= 180).astype(int)
n_stale = (visible_df["stale_180"] == 1).sum()
n_fresh = (visible_df["stale_180"] == 0).sum()
print(f"At FlyRank's exact threshold (180 days):")
print(f"  Stale+visible: n = {n_stale} — too few for a reliable verdict (floor ~50)")
print(f"  Fresh+visible: n = {n_fresh:,}")
print()

# Use the 91-day boundary instead, where we have solid n
visible_df["stale_91"] = (visible_df["days_since_last_update"] >= 91).astype(int)
rate_stale = visible_df[visible_df["stale_91"] == 1]["is_declining"].mean()
rate_fresh = visible_df[visible_df["stale_91"] == 0]["is_declining"].mean()
n_stale91 = (visible_df["stale_91"] == 1).sum()
n_fresh91 = (visible_df["stale_91"] == 0).sum()
print(f"At a testable threshold (91 days, same concept, solid sample):")
print(f"  Stale (≥91d) + visible: decline rate = {rate_stale*100:.1f}%, n = {n_stale91:,}")
print(f"  Fresh (<91d) + visible: decline rate = {rate_fresh*100:.1f}%, n = {n_fresh91:,}")
print(f"  Difference: {(rate_stale - rate_fresh)*100:+.1f} percentage points")
print()
print("Conclusion: The staleness signal is directionally CONFIRMED among visible")
print("pages at the 91-day boundary — stale pages decline ~3–5pp more often than")
print("fresh ones. The exact 180-day threshold from the product flag has too few")
print("pages (n=17) for a reliable verdict in this dataset slice, but the underlying")
print("signal — that staleness elevates decline risk — holds at a testable threshold.")

Visible pages (impressions ≥ 500): n = 16,726
Base decline rate (visible): 59.6%

Decline rate by staleness tier among VISIBLE pages (impr ≥ 500):
                    n  n_declining  decline_rate_pct
freshness_tier                                      
0-30            10063         5862              58.3
31-90              88           46              52.3
91-180           6558         4037              61.6
181+               17           16              94.1

At FlyRank's exact threshold (180 days):
  Stale+visible: n = 17 — too few for a reliable verdict (floor ~50)
  Fresh+visible: n = 16,709

At a testable threshold (91 days, same concept, solid sample):
  Stale (≥91d) + visible: decline rate = 61.6%, n = 6,575
  Fresh (<91d) + visible: decline rate = 58.2%, n = 10,151
  Difference: +3.4 percentage points

Conclusion: The staleness signal is directionally CONFIRMED among visible
pages at the 91-day boundary — stale pages decline ~3–5pp more often than
fresh ones. The exact 180-day

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [6]:
print("What a content team should take from this:")
print("=" * 60)
print()
print("1. Staleness is a real but imperfect signal. Pages not updated in 91-180")
print("   days decline more often than fresh pages, supporting the refresh flag's")
print("   logic. But very stale pages (181+) don't follow the pattern — they may")
print("   have already bottomed out. Prioritize the 91-180 day window for refreshes.")
print()
print("2. Position matters most in the striking-distance range (11-20). Pages that")
print("   were recently on page 1 and are slipping are the highest-risk group.")
print("   Deep pages (>50) are not good refresh candidates — they're already lost.")
print()
print("3. None of these signals are strong enough alone to build a decision rule.")
print("   The decline rates range from ~45-61% across buckets, against a 54% base")
print("   rate. A useful rule will need to combine multiple signals, and a model")
print("   that learns interactions may outperform any single-threshold rule.")

What a content team should take from this:

1. Staleness is a real but imperfect signal. Pages not updated in 91-180
   days decline more often than fresh pages, supporting the refresh flag's
   logic. But very stale pages (181+) don't follow the pattern — they may
   have already bottomed out. Prioritize the 91-180 day window for refreshes.

2. Position matters most in the striking-distance range (11-20). Pages that
   were recently on page 1 and are slipping are the highest-risk group.
   Deep pages (>50) are not good refresh candidates — they're already lost.

3. None of these signals are strong enough alone to build a decision rule.
   The decline rates range from ~45-61% across buckets, against a 54% base
   rate. A useful rule will need to combine multiple signals, and a model
   that learns interactions may outperform any single-threshold rule.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.